In [ ]:
"""
================================================================================
LEVEL 2 - TASK 2: DECISION TREE CLASSIFIER
================================================================================
Objective: Classify iris species using decision tree with pruning
Dataset: Iris Dataset
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("="*80)
print("LEVEL 2 - TASK 2: DECISION TREE CLASSIFIER")
print("="*80)

# ============================================================================
# STEP 1: SETUP PATHS
# ============================================================================

current_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
main_dir = os.path.dirname(current_dir)
dataset_path = os.path.join(main_dir, 'datasets', '1) iris.csv')
output_dir = os.path.join(main_dir, 'outputs')
image_dir = os.path.join(main_dir, 'images')

os.makedirs(output_dir, exist_ok=True)
os.makedirs(image_dir, exist_ok=True)

print(f"📁 Dataset path: {dataset_path}")

# ============================================================================
# STEP 2: LOAD AND PREPARE DATA
# ============================================================================

print("\n" + "="*80)
print("STEP 2: LOAD AND PREPARE DATA")
print("="*80)

# Load dataset
if os.path.exists(dataset_path):
    iris_df = pd.read_csv(dataset_path)
    print(f"✅ Loaded from: {dataset_path}")
else:
    alt_paths = ['../datasets/1) iris.csv', './datasets/1) iris.csv', '1) iris.csv']
    for path in alt_paths:
        if os.path.exists(path):
            iris_df = pd.read_csv(path)
            print(f"✅ Loaded from: {path}")
            break

# Encode target
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(iris_df['species'])

# Features
features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
X = iris_df[features]

print(f"\n📌 DATASET SHAPE: {X.shape[0]} rows × {X.shape[1]} features")
print(f"📌 Classes: {label_encoder.classes_}")

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📌 Training set: {X_train.shape[0]} samples")
print(f"📌 Testing set: {X_test.shape[0]} samples")

# ============================================================================
# STEP 3: TRAIN DECISION TREE WITHOUT PRUNING
# ============================================================================

print("\n" + "="*80)
print("STEP 3: TRAIN WITHOUT PRUNING")
print("="*80)

tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

y_pred_full = tree_full.predict(X_test)
accuracy_full = accuracy_score(y_test, y_pred_full)

print(f"✅ Model trained!")
print(f"   • Accuracy: {accuracy_full:.4f}")
print(f"   • Tree depth: {tree_full.get_depth()}")
print(f"   • Number of leaves: {tree_full.get_n_leaves()}")

# ============================================================================
# STEP 4: PRUNE TREE - FIND OPTIMAL DEPTH
# ============================================================================

print("\n" + "="*80)
print("STEP 4: PRUNE TREE - FIND OPTIMAL DEPTH")
print("="*80)

depths = [2, 3, 4, 5, 6, 7, 8, None]
train_accuracies = []
test_accuracies = []

print("\n📌 Testing different tree depths:")
print("-" * 50)
print(f"{'Depth':<10} {'Train Acc':<12} {'Test Acc':<12}")
print("-" * 50)

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    
    train_acc = tree.score(X_train, y_train)
    test_acc = tree.score(X_test, y_test)
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    
    depth_str = str(depth) if depth else "Unlimited"
    print(f"{depth_str:<10} {train_acc:.4f}       {test_acc:.4f}")

# Find best depth
best_idx = np.argmax(test_accuracies)
best_depth = depths[best_idx]
best_test_acc = test_accuracies[best_idx]

print(f"\n🏆 Best depth: {best_depth if best_depth else 'Unlimited'} (Test accuracy: {best_test_acc:.4f})")

# ============================================================================
# STEP 5: TRAIN PRUNED TREE
# ============================================================================

print("\n" + "="*80)
print("STEP 5: TRAIN PRUNED TREE")
print("="*80)

if best_depth:
    tree_pruned = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
else:
    tree_pruned = DecisionTreeClassifier(random_state=42)

tree_pruned.fit(X_train, y_train)
y_pred_pruned = tree_pruned.predict(X_test)
accuracy_pruned = accuracy_score(y_test, y_pred_pruned)

print(f"✅ Pruned tree trained!")
print(f"   • Accuracy: {accuracy_pruned:.4f}")
print(f"   • Tree depth: {tree_pruned.get_depth()}")
print(f"   • Number of leaves: {tree_pruned.get_n_leaves()}")

# ============================================================================
# STEP 6: FEATURE IMPORTANCE
# ============================================================================

print("\n" + "="*80)
print("STEP 6: FEATURE IMPORTANCE")
print("="*80)

feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': tree_pruned.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n📌 FEATURE IMPORTANCE:")
print(feature_importance)
print(f"\n📌 Most important: {feature_importance.iloc[0]['Feature']}")

# ============================================================================
# STEP 7: EVALUATE
# ============================================================================

print("\n" + "="*80)
print("STEP 7: EVALUATE PRUNED TREE")
print("="*80)

print("\n📌 CONFUSION MATRIX:")
cm = confusion_matrix(y_test, y_pred_pruned)
print(cm)

print("\n📌 CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_pruned, target_names=label_encoder.classes_))

# ============================================================================
# STEP 8: VISUALIZE RESULTS
# ============================================================================

print("\n" + "="*80)
print("STEP 8: VISUALIZE RESULTS")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Decision Tree Structure
plt.subplot(2, 2, 1)
plot_tree(tree_pruned, feature_names=features, class_names=label_encoder.classes_,
          filled=True, rounded=True, fontsize=10, max_depth=3)
plt.title(f'Decision Tree (max_depth={best_depth})')

# 2. Confusion Matrix
plt.subplot(2, 2, 2)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')

# 3. Feature Importance
plt.subplot(2, 2, 3)
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='green')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')

# 4. Depth vs Accuracy
plt.subplot(2, 2, 4)
depth_values = [str(d) if d else 'Inf' for d in depths]
plt.plot(depth_values, train_accuracies, 'b-o', label='Training', linewidth=2)
plt.plot(depth_values, test_accuracies, 'r-s', label='Testing', linewidth=2)
plt.xlabel('Tree Depth')
plt.ylabel('Accuracy')
plt.title('Training vs Testing Accuracy by Depth')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(image_dir, 'decision_tree_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {os.path.join(image_dir, 'decision_tree_results.png')}")

# ============================================================================
# STEP 9: SAVE RESULTS
# ============================================================================

print("\n" + "="*80)
print("STEP 9: SAVE RESULTS")
print("="*80)

results_df = pd.DataFrame({
    'Actual': label_encoder.inverse_transform(y_test),
    'Predicted': label_encoder.inverse_transform(y_pred_pruned),
    'Correct': y_test == y_pred_pruned
})
results_df.to_csv(os.path.join(output_dir, 'decision_tree_results.csv'), index=False)
print(f"✅ Saved: {os.path.join(output_dir, 'decision_tree_results.csv')}")

# ============================================================================
# STEP 10: SUMMARY
# ============================================================================

print("\n" + "="*80)
print("STEP 10: SUMMARY")
print("="*80)

print(f"""
╔════════════════════════════════════════════════════════════════════════════╗
║                    LEVEL 2 - TASK 2 COMPLETED                             ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║  ✅ Decision Tree Classifier with Pruning                                 ║
║                                                                            ║
║  PERFORMANCE:                                                             ║
║  ──────────────────────────────────────────────────────────────────────── ║
║  • Without Pruning: {accuracy_full:.4f}                                   ║
║  • With Pruning:    {accuracy_pruned:.4f}                                 ║
║  • Best Depth:      {best_depth if best_depth else 'Unlimited'}           ║
║                                                                            ║
║  OUTPUT FILES:                                                            ║
║  ──────────────────────────────────────────────────────────────────────── ║
║  • outputs/decision_tree_results.csv                                      ║
║  • images/decision_tree_results.png                                       ║
║                                                                            ║
╚════════════════════════════════════════════════════════════════════════════╝
""")

print("\n🎉 DECISION TREE COMPLETED SUCCESSFULLY!")